## Local Inference on GPU
Model page: https://huggingface.co/annajuliaasf/gemma-4-e2b-tool-use-ptbr

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/annajuliaasf/gemma-4-e2b-tool-use-ptbr)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [2]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("unsloth/gemma-4-e2b-it-unsloth-bnb-4bit")
model = PeftModel.from_pretrained(base_model, "annajuliaasf/gemma-4-e2b-tool-use-ptbr")

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 50.7MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

In [8]:
import torch
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("annajuliaasf/gemma-4-e2b-tool-use-ptbr")
dev = next(model.parameters()).device

INSTRUCOES = """<instrucoes>
Você é um assistente de IA com acesso a um conjunto de ferramentas.
Seu objetivo é responder às perguntas do usuário de forma correta e útil.

Como proceder:

1. Analise o pedido do usuário e entenda a intenção.
2. Decida se precisa de ferramenta:
   - Se você consegue responder com o que já sabe, responda direto.
   - Se precisa de informação externa ou de executar uma ação, use uma ferramenta.
3. Se for usar uma ferramenta, escolha a mais adequada entre as disponíveis
   e extraia os argumentos a partir do pedido do usuário.
4. Para chamar uma ferramenta, escreva a chamada dentro de <tool_call>.
   O conteúdo deve ser um objeto JSON:

   <tool_call>
   {"nome_tool": "nome_exato_da_ferramenta", "argumentos": {"parametro": "valor"}}
   </tool_call>

5. O resultado da ferramenta será devolvido a você dentro de <tool_result>.
   Interprete esse resultado para responder ao pedido original.
6. Sua resposta final ao usuário deve estar sempre dentro de <final_answer>,
   em linguagem natural. Se a ferramenta retornou erro, explique isso ao usuário.

   <final_answer>
   Sua resposta ao usuário.
   </final_answer>
</instrucoes>"""

FERRAMENTAS = """<ferramentas>
nome:get_traffic_info, descrição: Retorna informações de trânsito para uma rota específica., parâmetros: [{'nome': 'origin', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'destination', 'tipo': 'string', 'obrigatorio': True}]
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [{'nome': 'symbol', 'tipo': 'string', 'obrigatorio': True}]
nome:book_hotel, descrição: Reserva um quarto de hotel com as opções especificadas., parâmetros: [{'nome': 'hotel_name', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_in', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_out', 'tipo': 'string', 'obrigatorio': True}]
nome:create_budget, descrição: Cria um orçamento com base nas receitas e despesas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]
</ferramentas>"""

mensagens = [
    {"role": "system", "content": INSTRUCOES + "\n\n" + FERRAMENTAS},
    {"role": "user",   "content": "Quero saber o valor atual da ação da Microsoft."},
]

texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
entrada = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(dev)

with torch.no_grad():
    saida = model.generate(**entrada, max_new_tokens=800, do_sample=False)

print(tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True))


<tool_call>
{"nome_tool": "get_stock_price", "argumentos": {"symbol": "MSFT"}}
</tool_call>


#### setup

In [9]:
def perguntar(pergunta, ferramentas_texto):
    mensagens = [
        {"role": "system", "content": INSTRUCOES + "\n\n" + ferramentas_texto},
        {"role": "user",   "content": pergunta},
    ]
    texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(dev)
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=800, do_sample=False)
    return tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True)


In [10]:
FERRAMENTAS_1 = """<ferramentas>
nome:send_email, descrição: Envia um e-mail para o destinatário especificado., parâmetros: [{'nome': 'to', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'subject', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'body', 'tipo': 'string', 'obrigatorio': True}]
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [{'nome': 'symbol', 'tipo': 'string', 'obrigatorio': True}]
nome:book_hotel, descrição: Reserva um quarto de hotel com as opções especificadas., parâmetros: [{'nome': 'hotel_name', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_in', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_out', 'tipo': 'string', 'obrigatorio': True}]
nome:create_budget, descrição: Cria um orçamento com base nas receitas e despesas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]
</ferramentas>"""

print(perguntar("Manda um email pro meu chefe avisando que vou chegar atrasado.", FERRAMENTAS_1))


<tool_call>
{"nome_tool": "send_email", "argumentos": {"to": "chefe@empresa.com", "subject": "Atraso na chegada", "body": "Olá, estou com um pequeno atraso na chegada hoje. Estarei lá em breve e aviso assim que chegar. Obrigado."}}
</tool_call>


In [11]:
print(perguntar("Me dá uma receita de bolo de chocolate.", FERRAMENTAS_1))

<final_answer>
Claro! Aqui está uma receita clássica e deliciosa de bolo de chocolate:

**Ingredientes:**

*   2 xícaras de farinha de trigo
*   1 ½ xícara de açúcar
*   ¾ xícara de cacau em pó (ou chocolate em pó)
*   1 colher de chá de bicarbonato de sódio
*   1 colher de chá de fermento em pó
*   1 colher de chá de sal
*   2 ovos grandes
*   1 xícara de leite
*   ½ xícara de óleo vegetal
*   1 colher de sopa de extrato de baunilha
*   1 xícara de água quente (ou café quente, para intensificar o sabor do chocolate)

**Modo de Preparo:**

1.  **Pré-aqueça** o forno a 180°C e unte e enfarinhe uma forma redonda de 20cm.
2.  Em uma tigela grande, misture a farinha, o açúcar, o cacau em pó, o bicarbonato de sódio, o fermento em pó e o sal. Misture bem com um fouet ou batedeira.
3.  Em outra tigela, bata os ovos, o leite, o óleo e a baunilha até que estejam bem misturados.
4.  Adicione a mistura líquida aos ingredientes secos e misture até ficar homogêneo.
5.  Por último, adicione a água q

In [33]:
def perguntar_base(pergunta, ferramentas_texto):
    mensagens = [
        {"role": "system", "content": INSTRUCOES + "\n\n" + ferramentas_texto},
        {"role": "user",   "content": pergunta},
    ]
    texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(dev)
    with torch.no_grad():
        with model.disable_adapter():
            saida = model.generate(**entrada, max_new_tokens=800, do_sample=False)
    return tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True)

In [12]:
FERRAMENTAS_2 = """<ferramentas>
nome:control_thermostat, descrição: Controla a temperatura de um termostato., parâmetros: [{'nome': 'temperature', 'tipo': 'number', 'obrigatorio': True}]
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [{'nome': 'symbol', 'tipo': 'string', 'obrigatorio': True}]
nome:book_hotel, descrição: Reserva um quarto de hotel com as opções especificadas., parâmetros: [{'nome': 'hotel_name', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_in', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_out', 'tipo': 'string', 'obrigatorio': True}]
nome:create_budget, descrição: Cria um orçamento com base nas receitas e despesas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]
</ferramentas>"""

print(perguntar("Aumenta uns 3 graus aí no termostato, por favor.", FERRAMENTAS_2))

<tool_call>
{"nome_tool": "control_thermostat", "argumentos": {"temperature": 25}}
</tool_call>


In [13]:
FERRAMENTAS_SKILL = """<ferramentas>
nome:reset_password, descrição: Redefine a senha de um usuário. Passos: 1) Verificar o e-mail cadastrado na conta. 2) Gerar um código de verificação temporário. 3) Enviar o código por e-mail e aguardar confirmação., parâmetros: [{'nome': 'email', 'tipo': 'string', 'obrigatorio': True}]
nome:process_return, descrição: Processa a devolução de um produto comprado. Passos: 1) Verificar se o pedido está dentro do prazo de devolução. 2) Confirmar o motivo informado pelo cliente. 3) Gerar o código de reembolso e notificar o cliente., parâmetros: [{'nome': 'order_id', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'motivo', 'tipo': 'string', 'obrigatorio': True}]
nome:renew_subscription, descrição: Renova a assinatura de um plano existente. Passos: 1) Verificar o plano atual do usuário. 2) Calcular o valor da renovação. 3) Confirmar o pagamento e estender a validade., parâmetros: [{'nome': 'plano', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'meses', 'tipo': 'integer', 'obrigatorio': True}]
nome:escalate_ticket, descrição: Escala um chamado de suporte para um nível superior. Passos: 1) Verificar a prioridade atual do chamado. 2) Identificar a equipe responsável pelo próximo nível. 3) Transferir o chamado e notificar o time., parâmetros: [{'nome': 'ticket_id', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'prioridade', 'tipo': 'string', 'obrigatorio': True}]
</ferramentas>"""

print(perguntar("Esqueci minha senha, como faço pra recuperar o acesso?", FERRAMENTAS_SKILL))

<tool_call>
{"nome_tool": "reset_password", "argumentos": {"email": "usuario@exemplo.com"}}
</tool_call>


In [14]:
FERRAMENTAS_SEQUENCIA = """<ferramentas>
nome:check_account_balance, descrição: Verifica o saldo disponível em uma conta bancária. Deve ser chamada antes de qualquer operação que dependa do valor atual disponível, como criar um orçamento ou fazer um pagamento., parâmetros: [{'nome': 'account_id', 'tipo': 'string', 'obrigatorio': True}]
nome:create_budget, descrição: Cria um orçamento com base nas receitas e despesas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [{'nome': 'symbol', 'tipo': 'string', 'obrigatorio': True}]
nome:book_hotel, descrição: Reserva um quarto de hotel com as opções especificadas., parâmetros: [{'nome': 'hotel_name', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_in', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_out', 'tipo': 'string', 'obrigatorio': True}]
</ferramentas>"""

print(perguntar("Verifica meu saldo e depois monta um orçamento com esse valor pra esse mês.", FERRAMENTAS_SEQUENCIA))


<tool_call>
{"nome_tool": "check_account_balance", "argumentos": {"account_id": "123456789"}}
</tool_call>


In [15]:
INSTRUCOES_MULTIPASSO = INSTRUCOES.replace(
    "</instrucoes>",
    """
Algumas tarefas podem exigir mais de uma chamada de ferramenta em sequência,
quando o resultado de uma é necessário para executar a próxima. Nesses casos,
chame apenas a próxima ferramenta necessária e aguarde o resultado antes de
continuar — nunca invente o resultado de uma chamada anterior.
</instrucoes>"""
)

def perguntar_multipasso(pergunta, ferramentas_texto):
    mensagens = [
        {"role": "system", "content": INSTRUCOES_MULTIPASSO + "\n\n" + ferramentas_texto},
        {"role": "user",   "content": pergunta},
    ]
    texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entrada = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(dev)
    with torch.no_grad():
        saida = model.generate(**entrada, max_new_tokens=800, do_sample=False)
    return tokenizer.decode(saida[0][entrada["input_ids"].shape[1]:], skip_special_tokens=True)


print(perguntar_multipasso(
    "Verifica meu saldo e depois monta um orçamento com esse valor pra esse mês.",
    FERRAMENTAS_SEQUENCIA
))


<tool_call>
{"nome_tool": "check_account_balance", "argumentos": {"account_id": "123456789"}}
</tool_call>


In [16]:
def tool_str(nome, descricao, params):
    params_str = ", ".join(
        f"{{'nome': '{p[0]}', 'tipo': '{p[1]}', 'obrigatorio': True}}" for p in params
    )
    return f"nome:{nome}, descrição: {descricao}, parâmetros: [{params_str}]"

def montar_ferramentas(*tools):
    return "<ferramentas>\n" + "\n".join(tool_str(*t) for t in tools) + "\n</ferramentas>"


TESTES_SEQUENCIA = [
    ("Verifica se tem voo disponível de São Paulo pra Salvador dia 20 e depois reserva.",
     "check_flight_availability",
     [("check_flight_availability", "Verifica a disponibilidade de voos entre duas cidades numa data. Deve ser chamada antes de reservar um voo.", [("origin", "string"), ("destination", "string"), ("date", "string")]),
      ("book_flight", "Reserva um voo com as opções especificadas.", [("origin", "string"), ("destination", "string"), ("departure_date", "string")])]),

    ("Vê se tem quarto disponível no Hotel Central em dezembro e já reserva.",
     "check_room_availability",
     [("check_room_availability", "Verifica a disponibilidade de quartos num hotel para um período. Deve ser chamada antes de reservar um quarto.", [("hotel_name", "string"), ("date", "string")]),
      ("book_hotel", "Reserva um quarto de hotel com as opções especificadas.", [("hotel_name", "string"), ("check_in", "string"), ("check_out", "string")])]),

    ("Busca o histórico médico do paciente Carlos e cria um novo prontuário com isso.",
     "get_medical_history",
     [("get_medical_history", "Busca o histórico médico de um paciente pelo nome. Deve ser chamada antes de criar um novo prontuário.", [("patient_name", "string")]),
      ("create_medical_record", "Cria um novo registro médico com as informações especificadas.", [("patient_name", "string"), ("medical_history", "string")])]),

    ("Confere se tem estoque do produto X e gera a fatura pro cliente.",
     "verify_stock_availability",
     [("verify_stock_availability", "Verifica se há estoque disponível de um produto. Deve ser chamada antes de gerar uma fatura.", [("product_name", "string")]),
      ("generate_invoice", "Gera uma fatura com base nos itens comprados.", [("items", "array"), ("customer_info", "object")])]),

    ("Pega a cotação do dólar hoje e monta um orçamento em reais com base nisso.",
     "get_exchange_rate",
     [("get_exchange_rate", "Retorna a cotação atual entre duas moedas. Deve ser chamada antes de criar um orçamento em outra moeda.", [("currency_pair", "string")]),
      ("create_budget", "Cria um orçamento com base nas receitas e despesas.", [("income", "number"), ("expenses", "array")])]),

    ("Vê se o horário das 15h tá livre na agenda de amanhã e marca a reunião.",
     "check_calendar_availability",
     [("check_calendar_availability", "Verifica se um horário está livre na agenda. Deve ser chamada antes de agendar uma reunião.", [("date", "string"), ("time", "string")]),
      ("schedule_meeting", "Agenda uma reunião com os participantes especificados.", [("title", "string"), ("date", "string"), ("participants", "array")])]),

    ("Confirma a identidade do usuário 4521 e manda um SMS de confirmação pra ele.",
     "verify_user_identity",
     [("verify_user_identity", "Confirma a identidade de um usuário pelo ID. Deve ser chamada antes de enviar uma mensagem sensível.", [("user_id", "integer")]),
      ("send_sms", "Envia uma mensagem de texto para o número especificado.", [("number", "string"), ("message", "string")])]),

    ("Verifica o status da entrega do pedido #789 e gera um relatório com isso.",
     "check_delivery_status",
     [("check_delivery_status", "Verifica o status atual da entrega de um pedido. Deve ser chamada antes de gerar um relatório sobre a entrega.", [("order_id", "string")]),
      ("generate_report", "Gera um relatório com base nos dados especificados.", [("data", "array"), ("format", "string")])]),

    ("Vê como está o andamento do projeto Delta e registra minhas horas trabalhadas nele hoje.",
     "get_project_status",
     [("get_project_status", "Retorna o andamento atual de um projeto pelo nome. Deve ser chamada antes de registrar horas trabalhadas nesse projeto.", [("project_name", "string")]),
      ("track_work_hours", "Rastreia as horas trabalhadas em um projeto.", [("project_name", "string"), ("hours_worked", "number")])]),

    ("Confirma se o pagamento do plano premium caiu e renova minha assinatura por mais 3 meses.",
     "verify_payment",
     [("verify_payment", "Verifica se um pagamento foi confirmado. Deve ser chamada antes de renovar uma assinatura.", [("subscription_id", "string")]),
      ("renew_subscription", "Renova a assinatura de um plano existente.", [("plano", "string"), ("meses", "integer")])]),
]

for pergunta, primeiro_passo, tools in TESTES_SEQUENCIA:
    ferramentas = montar_ferramentas(*tools)
    print(f"PERGUNTA: {pergunta}")
    print(f"(esperado: chamar só '{primeiro_passo}')")
    print(perguntar(pergunta, ferramentas))
    print("-" * 60)

PERGUNTA: Verifica se tem voo disponível de São Paulo pra Salvador dia 20 e depois reserva.
(esperado: chamar só 'check_flight_availability')
<tool_call>
{"nome_tool": "check_flight_availability", "argumentos": {"origin": "S\u00e3o Paulo", "destination": "Salvador", "date": "2024-05-20"}}
</tool_call>
------------------------------------------------------------
PERGUNTA: Vê se tem quarto disponível no Hotel Central em dezembro e já reserva.
(esperado: chamar só 'check_room_availability')
<tool_call>
{"nome_tool": "check_room_availability", "argumentos": {"hotel_name": "Hotel Central", "date": "dezembro"}}
</tool_call>
------------------------------------------------------------
PERGUNTA: Busca o histórico médico do paciente Carlos e cria um novo prontuário com isso.
(esperado: chamar só 'get_medical_history')
<tool_call>
{"nome_tool": "get_medical_history", "argumentos": {"patient_name": "Carlos"}}
</tool_call>
------------------------------------------------------------
PERGUNTA: Conf

#### Inverter

In [17]:
# Teste 1, invertido: book_flight aparece ANTES de check_flight_availability
ferramentas_1_invertida = montar_ferramentas(
    ("book_flight", "Reserva um voo com as opções especificadas.",
     [("origin", "string"), ("destination", "string"), ("departure_date", "string")]),
    ("check_flight_availability", "Verifica a disponibilidade de voos entre duas cidades numa data. Deve ser chamada antes de reservar um voo.",
     [("origin", "string"), ("destination", "string"), ("date", "string")]),
)
print("TESTE 1 INVERTIDO")
print(perguntar("Verifica se tem voo disponível de São Paulo pra Salvador dia 20 e depois reserva.", ferramentas_1_invertida))
print("-" * 60)

# Teste 3, invertido: create_medical_record aparece ANTES de get_medical_history
ferramentas_3_invertida = montar_ferramentas(
    ("create_medical_record", "Cria um novo registro médico com as informações especificadas.",
     [("patient_name", "string"), ("medical_history", "string")]),
    ("get_medical_history", "Busca o histórico médico de um paciente pelo nome. Deve ser chamada antes de criar um novo prontuário.",
     [("patient_name", "string")]),
)
print("TESTE 3 INVERTIDO")
print(perguntar("Busca o histórico médico do paciente Carlos e cria um novo prontuário com isso.", ferramentas_3_invertida))
print("-" * 60)


TESTE 1 INVERTIDO
<tool_call>
{"nome_tool": "check_flight_availability", "argumentos": {"origin": "S\u00e3o Paulo", "destination": "Salvador", "date": "2024-05-20"}}
</tool_call>
------------------------------------------------------------
TESTE 3 INVERTIDO
<tool_call>
{"nome_tool": "get_medical_history", "argumentos": {"patient_name": "Carlos"}}
</tool_call>
------------------------------------------------------------


### Modelo base

In [24]:
import os

if not os.path.exists("ft-llm-tool-calling"):
    !git clone https://github.com/annaferreiras/ft-llm-tool-calling.git

# confere que os arquivos existem agora
import json
testes = [json.loads(l) for l in open("ft-llm-tool-calling/dados/teste.jsonl", encoding="utf-8")]
saidas_base_originais = json.load(open("ft-llm-tool-calling/resultados/saidas_base.json", encoding="utf-8"))

Cloning into 'ft-llm-tool-calling'...
remote: Enumerating objects: 49, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 49 (delta 18), reused 38 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (49/49), 290.20 KiB | 14.51 MiB/s, done.
Resolving deltas: 100% (18/18), done.


In [27]:
import json
testes = [json.loads(l) for l in open("ft-llm-tool-calling/dados/teste.jsonl", encoding="utf-8")]
saidas_base_originais = json.load(open("ft-llm-tool-calling/resultados/saidas_base.json", encoding="utf-8"))

pergunta_original = testes[3]["messages"][1]["content"]
ferramentas_original = "<ferramentas>" + testes[3]["messages"][0]["content"].split("<ferramentas>")[1]

print("Pergunta:", pergunta_original)
print()
print("Saída SALVA do base, já documentada (deve ser sem tag, prosa comum):")
print(saidas_base_originais[3])
print()
print("Saída do disable_adapter() agora, mesma pergunta e mesmas ferramentas:")
print(perguntar_base(pergunta_original, ferramentas_original))

Pergunta: Posso ter um orçamento para as minhas despesas?

Saída SALVA do base, já documentada (deve ser sem tag, prosa comum):
Para que eu possa criar um orçamento para suas despesas, preciso que você me forneça as informações sobre suas receitas (entradas de dinheiro) e suas despesas (saídas de dinheiro).

Você pode me dizer quanto você ganha e quais são suas despesas? Por exemplo:

*   **Receitas:** (Ex: Salário, renda extra)
*   **Despesas:** (Ex: Aluguel, contas de consumo, alimentação, lazer, etc.)

Assim que você me fornecer esses dados, eu posso usar a ferramenta `create_budget` para criar o orçamento para você!

Saída do disable_adapter() agora, mesma pergunta e mesmas ferramentas:
Para que eu possa criar um orçamento para suas despesas, preciso que você me forneça as informações sobre suas receitas (entradas de dinheiro) e suas despesas (saídas de dinheiro).

Você pode me dizer quanto você ganha e quais são suas despesas? Por exemplo:

*   **Receitas:** R$ 5000,00
*   **Despe

In [28]:
TESTES_COMPARACAO = [
    ("Alguém sabe me dizer quanto está a ação da Apple hoje?", FERRAMENTAS_1),

    ("Me dá uma receita de bolo de chocolate.", FERRAMENTAS_1),

    ("Esqueci minha senha, como faço pra recuperar o acesso?", FERRAMENTAS_SKILL),

    ("Verifica se tem voo disponível de São Paulo pra Salvador dia 20 e depois reserva.",
     montar_ferramentas(
         ("check_flight_availability", "Verifica a disponibilidade de voos entre duas cidades numa data. Deve ser chamada antes de reservar um voo.",
          [("origin", "string"), ("destination", "string"), ("date", "string")]),
         ("book_flight", "Reserva um voo com as opções especificadas.",
          [("origin", "string"), ("destination", "string"), ("departure_date", "string")]),
     )),
]

for pergunta, ferramentas in TESTES_COMPARACAO:
    print(f"PERGUNTA: {pergunta}")
    print("COM fine-tuning:")
    print(perguntar(pergunta, ferramentas))
    print()
    print("SEM fine-tuning (base):")
    print(perguntar_base(pergunta, ferramentas))
    print("-" * 60)


PERGUNTA: Alguém sabe me dizer quanto está a ação da Apple hoje?
COM fine-tuning:
<tool_call>
{"nome_tool": "get_stock_price", "argumentos": {"symbol": "AAPL"}}
</tool_call>

SEM fine-tuning (base):
<tool_call>
{"nome_tool": "get_stock_price", "argumentos": {"symbol": "AAPL"}}
</tool_call>
------------------------------------------------------------
PERGUNTA: Me dá uma receita de bolo de chocolate.
COM fine-tuning:
<final_answer>
Claro! Aqui está uma receita clássica e deliciosa de bolo de chocolate:

**Ingredientes:**

*   2 xícaras de farinha de trigo
*   1 ½ xícara de açúcar
*   ¾ xícara de cacau em pó (ou chocolate em pó)
*   1 colher de chá de bicarbonato de sódio
*   1 colher de chá de fermento em pó
*   1 colher de chá de sal
*   2 ovos grandes
*   1 xícara de leite
*   ½ xícara de óleo vegetal
*   1 colher de sopa de extrato de baunilha
*   1 xícara de água quente (ou café quente, para intensificar o sabor do chocolate)

**Modo de Preparo:**

1.  **Pré-aqueça** o forno a 180°C 

In [29]:
TESTES_BASE = [
    ("Alguém sabe me dizer quanto está a ação da Apple hoje?", FERRAMENTAS_1),

    ("Me dá uma receita de bolo de chocolate.", FERRAMENTAS_1),

    ("Aumenta uns 3 graus aí no termostato, por favor.", FERRAMENTAS_2),

    ("Esqueci minha senha, como faço pra recuperar o acesso?", FERRAMENTAS_SKILL),

    ("Verifica meu saldo e depois monta um orçamento com esse valor pra esse mês.", FERRAMENTAS_SEQUENCIA),

    ("Verifica se tem voo disponível de São Paulo pra Salvador dia 20 e depois reserva.",
     montar_ferramentas(
         ("check_flight_availability", "Verifica a disponibilidade de voos entre duas cidades numa data. Deve ser chamada antes de reservar um voo.", [("origin", "string"), ("destination", "string"), ("date", "string")]),
         ("book_flight", "Reserva um voo com as opções especificadas.", [("origin", "string"), ("destination", "string"), ("departure_date", "string")]))),

    ("Vê se tem quarto disponível no Hotel Central em dezembro e já reserva.",
     montar_ferramentas(
         ("check_room_availability", "Verifica a disponibilidade de quartos num hotel para um período. Deve ser chamada antes de reservar um quarto.", [("hotel_name", "string"), ("date", "string")]),
         ("book_hotel", "Reserva um quarto de hotel com as opções especificadas.", [("hotel_name", "string"), ("check_in", "string"), ("check_out", "string")]))),

    ("Busca o histórico médico do paciente Carlos e cria um novo prontuário com isso.",
     montar_ferramentas(
         ("get_medical_history", "Busca o histórico médico de um paciente pelo nome. Deve ser chamada antes de criar um novo prontuário.", [("patient_name", "string")]),
         ("create_medical_record", "Cria um novo registro médico com as informações especificadas.", [("patient_name", "string"), ("medical_history", "string")]))),

    ("Confere se tem estoque do produto X e gera a fatura pro cliente.",
     montar_ferramentas(
         ("verify_stock_availability", "Verifica se há estoque disponível de um produto. Deve ser chamada antes de gerar uma fatura.", [("product_name", "string")]),
         ("generate_invoice", "Gera uma fatura com base nos itens comprados.", [("items", "array"), ("customer_info", "object")]))),

    ("Pega a cotação do dólar hoje e monta um orçamento em reais com base nisso.",
     montar_ferramentas(
         ("get_exchange_rate", "Retorna a cotação atual entre duas moedas. Deve ser chamada antes de criar um orçamento em outra moeda.", [("currency_pair", "string")]),
         ("create_budget", "Cria um orçamento com base nas receitas e despesas.", [("income", "number"), ("expenses", "array")]))),

    ("Vê se o horário das 15h tá livre na agenda de amanhã e marca a reunião.",
     montar_ferramentas(
         ("check_calendar_availability", "Verifica se um horário está livre na agenda. Deve ser chamada antes de agendar uma reunião.", [("date", "string"), ("time", "string")]),
         ("schedule_meeting", "Agenda uma reunião com os participantes especificados.", [("title", "string"), ("date", "string"), ("participants", "array")]))),

    ("Confirma a identidade do usuário 4521 e manda um SMS de confirmação pra ele.",
     montar_ferramentas(
         ("verify_user_identity", "Confirma a identidade de um usuário pelo ID. Deve ser chamada antes de enviar uma mensagem sensível.", [("user_id", "integer")]),
         ("send_sms", "Envia uma mensagem de texto para o número especificado.", [("number", "string"), ("message", "string")]))),

    ("Verifica o status da entrega do pedido #789 e gera um relatório com isso.",
     montar_ferramentas(
         ("check_delivery_status", "Verifica o status atual da entrega de um pedido. Deve ser chamada antes de gerar um relatório sobre a entrega.", [("order_id", "string")]),
         ("generate_report", "Gera um relatório com base nos dados especificados.", [("data", "array"), ("format", "string")]))),

    ("Vê como está o andamento do projeto Delta e registra minhas horas trabalhadas nele hoje.",
     montar_ferramentas(
         ("get_project_status", "Retorna o andamento atual de um projeto pelo nome. Deve ser chamada antes de registrar horas trabalhadas nesse projeto.", [("project_name", "string")]),
         ("track_work_hours", "Rastreia as horas trabalhadas em um projeto.", [("project_name", "string"), ("hours_worked", "number")]))),

    ("Confirma se o pagamento do plano premium caiu e renova minha assinatura por mais 3 meses.",
     montar_ferramentas(
         ("verify_payment", "Verifica se um pagamento foi confirmado. Deve ser chamada antes de renovar uma assinatura.", [("subscription_id", "string")]),
         ("renew_subscription", "Renova a assinatura de um plano existente.", [("plano", "string"), ("meses", "integer")]))),
]

for pergunta, ferramentas in TESTES_BASE:
    print(f"PERGUNTA: {pergunta}")
    print(perguntar_base(pergunta, ferramentas))
    print("-" * 60)


PERGUNTA: Alguém sabe me dizer quanto está a ação da Apple hoje?
<tool_call>
{"nome_tool": "get_stock_price", "argumentos": {"symbol": "AAPL"}}
</tool_call>
------------------------------------------------------------
PERGUNTA: Me dá uma receita de bolo de chocolate.
<final_answer>
Claro! Aqui está uma receita clássica e deliciosa de bolo de chocolate:

**Ingredientes:**

*   **Para o bolo:**
    *   2 xícaras de farinha de trigo
    *   1 ½ xícara de açúcar
    *   ¾ xícara de cacau em pó (ou chocolate em pó, de boa qualidade)
    *   1 colher de chá de bicarbonato de sódio
    *   1 colher de chá de fermento em pó
    *   1 pitada de sal
    *   2 ovos grandes
    *   1 xícara de leite
    *   ½ xícara de óleo vegetal
    *   1 colher de chá de extrato de baunilha
    *   1 xícara de água quente (ou café quente, para intensificar o sabor do chocolate)

*   **Para a cobertura (opcional - Ganache Simples):**
    *   200g de chocolate meio amargo picado
    *   200ml de creme de leite



In [30]:
print("Saída SALVA do base, já documentada (deve ser sem tag, prosa comum):")
print(saidas_base_originais[3])
print()
print("Saída do disable_adapter() agora, mesma pergunta e mesmas ferramentas:")
print(perguntar_base(pergunta_original, ferramentas_original))


Saída SALVA do base, já documentada (deve ser sem tag, prosa comum):
Para que eu possa criar um orçamento para suas despesas, preciso que você me forneça as informações sobre suas receitas (entradas de dinheiro) e suas despesas (saídas de dinheiro).

Você pode me dizer quanto você ganha e quais são suas despesas? Por exemplo:

*   **Receitas:** (Ex: Salário, renda extra)
*   **Despesas:** (Ex: Aluguel, contas de consumo, alimentação, lazer, etc.)

Assim que você me fornecer esses dados, eu posso usar a ferramenta `create_budget` para criar o orçamento para você!

Saída do disable_adapter() agora, mesma pergunta e mesmas ferramentas:
Para que eu possa criar um orçamento para suas despesas, preciso que você me forneça as informações sobre suas receitas (entradas de dinheiro) e suas despesas (saídas de dinheiro).

Você pode me dizer quanto você ganha e quais são suas despesas? Por exemplo:

*   **Receitas:** R$ 5000,00
*   **Despesas:** Aluguel (R$ 1500), Alimentação (R$ 1000), Contas (R$

In [31]:
FERRAMENTAS_ORIGINAL = """<ferramentas>
nome:get_traffic_info, descrição: Retorna informações de trânsito para uma rota específica., parâmetros: [{'nome': 'origin', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'destination', 'tipo': 'string', 'obrigatorio': True}]
nome:get_stock_price, descrição: Retorna o preço atual de uma ação específica., parâmetros: [{'nome': 'symbol', 'tipo': 'string', 'obrigatorio': True}]
nome:book_hotel, descrição: Reserva um quarto de hotel com as opções especificadas., parâmetros: [{'nome': 'hotel_name', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_in', 'tipo': 'string', 'obrigatorio': True}, {'nome': 'check_out', 'tipo': 'string', 'obrigatorio': True}]
nome:create_budget, descrição: Cria um orçamento com base nas receitas e despesas., parâmetros: [{'nome': 'income', 'tipo': 'number', 'obrigatorio': True}, {'nome': 'expenses', 'tipo': 'array', 'obrigatorio': True}]
</ferramentas>"""

print(perguntar_base("Qual é o índice de UV em Fortaleza agora?", FERRAMENTAS_ORIGINAL))


<tool_call>
{"nome_tool": "get_traffic_info", "argumentos": {"origin": "Fortaleza", "destination": "Fortaleza"}}
</tool_call>


In [32]:
def tool_str(nome, descricao, params):
    params_str = ", ".join(
        f"{{'nome': '{p[0]}', 'tipo': '{p[1]}', 'obrigatorio': True}}" for p in params
    )
    return f"nome:{nome}, descrição: {descricao}, parâmetros: [{params_str}]"

def montar_ferramentas(*tools):
    return "<ferramentas>\n" + "\n".join(tool_str(*t) for t in tools) + "\n</ferramentas>"


TESTES_DADO_FALTANDO = [
    ("Rastreia minha encomenda.",
     [("track_package", "Rastreia o status de entrega de um pacote.", [("tracking_number", "string")])]),

    ("Me dá informação sobre um usuário.",
     [("get_user_info", "Retorna informações de um usuário específico.", [("user_id", "integer")])]),

    ("Cria um prontuário pro paciente que chegou agora.",
     [("create_medical_record", "Cria um novo registro médico com as informações especificadas.", [("patient_name", "string"), ("medical_history", "string")])]),

    ("Reserva um hotel pra mim.",
     [("book_hotel", "Reserva um quarto de hotel com as opções especificadas.", [("hotel_name", "string"), ("check_in", "string"), ("check_out", "string")])]),

    ("Marca uma reunião.",
     [("schedule_meeting", "Agenda uma reunião com os participantes especificados.", [("title", "string"), ("date", "string"), ("participants", "array")])]),

    ("Gera a fatura pro cliente de hoje.",
     [("generate_invoice", "Gera uma fatura com base nos itens comprados.", [("items", "array"), ("customer_info", "object")])]),

    ("Manda um email pro pessoal do RH perguntando sobre férias.",
     [("send_email", "Envia um e-mail para o destinatário especificado.", [("to", "string"), ("subject", "string"), ("body", "string")])]),

    ("Cria uma tarefa pra eu não esquecer de ligar pro cliente.",
     [("create_task", "Cria uma nova tarefa com os detalhes especificados.", [("title", "string"), ("description", "string")])]),
]

def tem_tag(saida):
    return ("<tool_call>" in saida and "</tool_call>" in saida) or ("<final_answer>" in saida and "</final_answer>" in saida)

resultados = []
for pergunta, tools in TESTES_DADO_FALTANDO:
    ferramentas = montar_ferramentas(*tools)
    saida_ft = perguntar(pergunta, ferramentas)
    saida_base = perguntar_base(pergunta, ferramentas)
    resultados.append((pergunta, tem_tag(saida_ft), tem_tag(saida_base)))
    print(f"PERGUNTA: {pergunta}")
    print(f"  fine-tuning tem tag? {tem_tag(saida_ft)}")
    print(f"  base tem tag?        {tem_tag(saida_base)}")
    print("  --- saída fine-tuning ---")
    print(saida_ft)
    print("  --- saída base ---")
    print(saida_base)
    print("-" * 60)

print()
print("RESUMO")
print(f"fine-tuning: {sum(r[1] for r in resultados)}/{len(resultados)} com tag")
print(f"base:        {sum(r[2] for r in resultados)}/{len(resultados)} com tag")


PERGUNTA: Rastreia minha encomenda.
  fine-tuning tem tag? True
  base tem tag?        False
  --- saída fine-tuning ---
<tool_call>
{"nome_tool": "track_package", "argumentos": {"tracking_number": "BR123456789BR"}}
</tool_call>
  --- saída base ---
Para que eu possa rastrear sua encomenda, por favor, me forneça o número de rastreamento.
------------------------------------------------------------
PERGUNTA: Me dá informação sobre um usuário.
  fine-tuning tem tag? True
  base tem tag?        False
  --- saída fine-tuning ---
<tool_call>
{"nome_tool": "get_user_info", "argumentos": {"user_id": 123}}
</tool_call>
  --- saída base ---
Para que eu possa te dar informações sobre um usuário, preciso que você me forneça o **ID do usuário**.

Você pode me dizer o `user_id`?
------------------------------------------------------------
PERGUNTA: Cria um prontuário pro paciente que chegou agora.
  fine-tuning tem tag? True
  base tem tag?        False
  --- saída fine-tuning ---
<tool_call>
{"nom